# P2P tensor communication

Use `torch.Tensor` payloads with `dist_puzzles.send`, `recv`, `isend`, and `irecv`.

In [1]:
# ruff: noqa: F401, F841

import flock
import torch

import dist_puzzles as dp


def same(actual, expected):
    if isinstance(expected, torch.Tensor):
        assert isinstance(actual, torch.Tensor)
        assert torch.equal(actual, expected), (actual, expected)
        return
    if isinstance(expected, (list, tuple)):
        assert len(actual) == len(expected), (actual, expected)
        for got, want in zip(actual, expected, strict=True):
            same(got, want)
        return
    assert actual == expected, (actual, expected)

## 1. Ping-pong

Rank 0 sends a tensor to rank 1. Rank 1 sends back that tensor plus one.

In [2]:
@flock.distribute(workers=2)
async def ping_pong():
    rank = flock.get_rank()
    if rank == 0:
        x = torch.tensor([41.0])
        await dp.send(1, x)
        return await dp.recv(1)

    x = await dp.recv(0)
    await dp.send(0, x + 1)
    return None


same(ping_pong(), [torch.tensor([42.0]), None])

## 2. Blocking exchange

Two ranks exchange tensors with blocking `send` / `recv` without deadlocking.

In [3]:
@flock.distribute(workers=2)
async def blocking_exchange():
    rank = flock.get_rank()
    x = torch.tensor([rank])

    if rank == 0:
        await dp.send(1, x)
        return await dp.recv(1)

    other = await dp.recv(0)
    await dp.send(0, x)
    return other


same(blocking_exchange(), [torch.tensor([1]), torch.tensor([0])])

## 3. Pair exchange by parity

Ranks exchange tensors with their pair: `0 ↔ 1`, `2 ↔ 3`.

In [4]:
@flock.distribute(workers=4)
async def parity_exchange():
    rank = flock.get_rank()
    partner = rank ^ 1
    x = torch.tensor([rank])

    if rank % 2 == 0:
        await dp.send(partner, x)
        return await dp.recv(partner)

    other = await dp.recv(partner)
    await dp.send(partner, x)
    return other


same(
    parity_exchange(),
    [torch.tensor([1]), torch.tensor([0]), torch.tensor([3]), torch.tensor([2])],
)

## 4. Nonblocking exchange

Use `isend` / `irecv` so every rank can post communication before waiting.

In [5]:
@flock.distribute(workers=2)
async def nonblocking_exchange():
    rank = flock.get_rank()
    peer = 1 - rank
    x = torch.tensor([rank])

    send_work = dp.isend(peer, x)
    recv_work = dp.irecv(peer)
    other = await recv_work.wait()
    await send_work.wait()
    return other


same(nonblocking_exchange(), [torch.tensor([1]), torch.tensor([0])])

## 5. Ring shift

Each rank sends its tensor to the next rank and receives from the previous rank.

In [6]:
@flock.distribute(workers=4)
async def ring_shift():
    rank = flock.get_rank()
    world = flock.get_world_size()
    dst = (rank + 1) % world
    src = (rank - 1) % world
    x = torch.tensor([rank])

    await dp.isend(dst, x).wait()
    value = await dp.recv(src)
    return value


same(
    ring_shift(),
    [torch.tensor([3]), torch.tensor([0]), torch.tensor([1]), torch.tensor([2])],
)

## 6. Ring token

A token starts at rank 0 and moves once around the ring.

In [7]:
@flock.distribute(workers=4)
async def ring_token():
    rank = flock.get_rank()
    world = flock.get_world_size()
    dst = (rank + 1) % world
    src = (rank - 1) % world

    if rank == 0:
        await dp.send(dst, torch.tensor([0]))
        token = await dp.recv(src)
        return token + 1

    token = await dp.recv(src)
    token = token + 1
    await dp.send(dst, token)
    return token


same(
    ring_token(),
    [torch.tensor([4]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])],
)

## 7. Tagged messages

Rank 0 sends two tensors to rank 1 with different tags. Rank 1 receives tag `1` before tag `0`.

In [8]:
@flock.distribute(workers=2)
async def tagged_messages():
    rank = flock.get_rank()
    if rank == 0:
        await dp.isend(1, torch.tensor([0]), tag=0).wait()
        await dp.isend(1, torch.tensor([1]), tag=1).wait()
        return None

    tag_one = await dp.recv(0, tag=1)
    tag_zero = await dp.recv(0, tag=0)
    return tag_one, tag_zero


same(tagged_messages(), [None, (torch.tensor([1]), torch.tensor([0]))])

## 8. Neighbor exchange

Each rank receives tensors from its left and right ring neighbors.

In [9]:
@flock.distribute(workers=4)
async def neighbor_exchange():
    rank = flock.get_rank()
    world = flock.get_world_size()
    left = (rank - 1) % world
    right = (rank + 1) % world
    x = torch.tensor([rank])

    await dp.isend(left, x, tag=0).wait()
    await dp.isend(right, x, tag=1).wait()
    from_left = await dp.recv(left, tag=1)
    from_right = await dp.recv(right, tag=0)
    return from_left, from_right


same(
    neighbor_exchange(),
    [
        (torch.tensor([3]), torch.tensor([1])),
        (torch.tensor([0]), torch.tensor([2])),
        (torch.tensor([1]), torch.tensor([3])),
        (torch.tensor([2]), torch.tensor([0])),
    ],
)

## 9. Any-source receive

Ranks 1, 2, and 3 send to rank 0. Rank 0 receives three tensors with `flock.ANY_SOURCE`.

In [10]:
@flock.distribute(workers=4)
async def any_source_gather():
    rank = flock.get_rank()
    if rank != 0:
        await dp.send(0, torch.tensor([rank]))
        return None

    values = []
    for _ in range(3):
        values.append(await dp.recv(flock.ANY_SOURCE))
    return values


result = any_source_gather()
same(result[1:], [None, None, None])
same(sorted(x.item() for x in result[0]), [1, 2, 3])

## 10. Manual gather to root

Every non-root rank sends its tensor to rank 0. Rank 0 returns tensors ordered by rank.

In [11]:
@flock.distribute(workers=4)
async def manual_gather():
    rank = flock.get_rank()
    x = torch.tensor([rank])
    if rank == 0:
        values = [x]
        for src in range(1, flock.get_world_size()):
            values.append(await dp.recv(src))
        return values

    await dp.isend(0, x).wait()
    return None


same(
    manual_gather(),
    [[torch.tensor([0]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])], None, None, None],
)

## 14. All-to-all

Each rank splits a local tensor into one slice per destination, then receives one slice from every source.

In [12]:
@flock.distribute(workers=4, policy=flock.Fifo())
async def all_to_all_slices():
    rank = flock.get_rank()
    world = flock.get_world_size()
    x = torch.arange(rank * 10, rank * 10 + world)

    print(x)

    sends = []
    for dst in range(world):
        sends.append(dp.isend(dst, x[dst : dst + 1]))

    values = []
    for src in range(world):
        values.append(await dp.recv(src))

    for work in sends:
        await work.wait()

    return sorted(values, key=lambda value: value.item())


same(
    all_to_all_slices(),
    [
        [torch.tensor([0]), torch.tensor([10]), torch.tensor([20]), torch.tensor([30])],
        [torch.tensor([1]), torch.tensor([11]), torch.tensor([21]), torch.tensor([31])],
        [torch.tensor([2]), torch.tensor([12]), torch.tensor([22]), torch.tensor([32])],
        [torch.tensor([3]), torch.tensor([13]), torch.tensor([23]), torch.tensor([33])],
    ],
)

tensor([0, 1, 2, 3])
tensor([10, 11, 12, 13])
tensor([20, 21, 22, 23])
tensor([30, 31, 32, 33])


## 15. Pipeline chain

Three microbatches move through the rank chain. Each nonzero rank adds one before forwarding.

In [13]:
@flock.distribute(workers=4)
async def pipeline_chain():
    rank = flock.get_rank()
    world = flock.get_world_size()
    values = []

    for microbatch in range(3):
        if rank == 0:
            value = torch.tensor([microbatch])
        else:
            value = await dp.recv(rank - 1)
            value = value + 1

        if rank != world - 1:
            await dp.send(rank + 1, value)

        values.append(value)

    return values


same(
    pipeline_chain(),
    [
        [torch.tensor([0]), torch.tensor([1]), torch.tensor([2])],
        [torch.tensor([1]), torch.tensor([2]), torch.tensor([3])],
        [torch.tensor([2]), torch.tensor([3]), torch.tensor([4])],
        [torch.tensor([3]), torch.tensor([4]), torch.tensor([5])],
    ],
)

## 16. Max-rank election

Circulate a candidate around the ring and return the elected rank.

In [14]:
values = [10, 3, 2, 42]

@flock.distribute(workers=4)
async def max_rank_election():
    rank = flock.get_rank()
    world = flock.get_world_size()
    dst = (rank + 1) % world
    src = (rank - 1) % world

    if rank == 0:
        await dp.send(dst, torch.tensor([values[rank]]))
        elected = await dp.recv(src)
        await dp.send(dst, elected)
        return elected

    candidate = await dp.recv(src)
    candidate = torch.maximum(candidate, torch.tensor([rank]))
    await dp.send(dst, candidate)
    elected = await dp.recv(src)
    if rank != world - 1:
        await dp.send(dst, elected)
    return elected


same(max_rank_election(), [torch.tensor([10]), torch.tensor([10]), torch.tensor([10]), torch.tensor([10])])